Instalacion de librerias

In [ ]:
!pip install opendatasets pandas scikit-learn numpy


Subir a Colab

In [ ]:
from google.colab import files
files.upload()

{}

Configurar acceso

In [ ]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


DESCARGAR DATA SET

In [ ]:
import opendatasets as od

od.download("https://www.kaggle.com/datasets/stealthtechnologies/car-evaluation-classification")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: yovaniticona
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/stealthtechnologies/car-evaluation-classification


100%|██████████| 4.71k/4.71k [00:00<00:00, 8.36MB/s]

CARGAR DATASET Y PREPROCESAR

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("car-evaluation-classification/cars.csv")

# Convertir texto a números
le = LabelEncoder()
for col in df.columns:
    df[col] = le.fit_transform(df[col])

X = df.drop("class", axis=1).values
y = df["class"].values
feature_names = df.columns[:-1]

print(df.head())

   buying  maint  doors  persons  lug_boot  safety  class
0       3      3      0        0         2       1      2
1       3      3      0        0         2       2      2
2       3      3      0        0         2       0      2
3       3      3      0        0         1       1      2
4       3      3      0        0         1       2      2


ALGORITMO GENÉTICO ADAPTADO

In [ ]:
import numpy as np
import random
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

# Configuración
random.seed(42)
np.random.seed(42)

# Escalar datos
X = StandardScaler().fit_transform(X)
NUM_FEATURES = X.shape[1]

# Crear cromosoma
def crear():
    while True:
        c = [random.randint(0,1) for _ in range(NUM_FEATURES)]
        if sum(c) > 0:
            return c

# Fitness
def fitness(c):
    idx = [i for i,g in enumerate(c) if g==1]
    if not idx:
        return 0
    X_sub = X[:, idx]
    model = LogisticRegression(max_iter=200)
    acc = cross_val_score(model, X_sub, y, cv=5).mean()
    return acc - 0.01*(len(idx)/NUM_FEATURES)

# Inicializar
poblacion = [crear() for _ in range(20)]

# Evolución
for gen in range(30):
    apt = [fitness(c) for c in poblacion]
    nueva = [poblacion[np.argmax(apt)]]

    while len(nueva) < 20:
        p1 = random.choice(poblacion)
        p2 = random.choice(poblacion)

        punto = random.randint(1, NUM_FEATURES-1)
        h = p1[:punto] + p2[punto:]

        # Mutación
        for i in range(NUM_FEATURES):
            if random.random() < 0.05:
                h[i] = 1 - h[i]

        nueva.append(h)

    poblacion = nueva

# Resultado final
mejor = max(poblacion, key=fitness)
seleccionadas = [feature_names[i] for i,g in enumerate(mejor) if g==1]

print("\nFeatures seleccionadas:")
for f in seleccionadas:
    print("✓", f)


Features seleccionadas:
✓ persons
✓ safety


In [ ]:
"""
=============================================================
  ALGORITMO GENÉTICO PARA HYPERPARAMETER OPTIMIZATION
=============================================================
  Objetivo: Encontrar los mejores hiperparámetros para un
  modelo Random Forest usando un Algoritmo Genético.
  Dataset: Wine (sklearn) — clasificación multiclase
=============================================================
"""

import numpy as np
import random
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

# ─── Configuración general ────────────────────────────────
RANDOM_SEED      = 42
TAM_POBLACION    = 16
NUM_GENERACIONES = 25
TASA_CRUCE       = 0.85
TASA_MUTACION    = 0.10   # Mayor que feature-selection (espacio continuo)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ─── Carga del dataset ────────────────────────────────────
datos = load_wine()
X, y  = datos.data, datos.target
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("=" * 60)
print("  AG PARA HYPERPARAMETER OPTIMIZATION — Wine Dataset")
print("=" * 60)
print(f"  Muestras    : {X.shape[0]}")
print(f"  Features    : {X.shape[1]}")
print(f"  Clases      : {len(np.unique(y))}")
print("=" * 60)

# ─── 1. REPRESENTACIÓN ────────────────────────────────────
# Espacio de hiperparámetros a optimizar (Random Forest)
ESPACIO = {
    'n_estimators' : [50, 100, 150, 200, 300],   # árboles
    'max_depth'    : [None, 5, 10, 15, 20],       # profundidad
    'min_samples_split': [2, 5, 10, 15],          # mínimo muestras para split
    'min_samples_leaf' : [1, 2, 4, 8],            # mínimo en hojas
    'max_features' : ['sqrt', 'log2', None],      # features por árbol
}

LLAVES = list(ESPACIO.keys())
LONGITUDES = [len(ESPACIO[k]) for k in LLAVES]

# Cromosoma: lista de índices, uno por cada hiperparámetro
# Ej: [2, 0, 1, 3, 1] → índice en cada lista de opciones

def cromosoma_a_hiperparametros(cromosoma):
    """Convierte un cromosoma (índices) a un dict de hiperparámetros."""
    return {llave: ESPACIO[llave][cromosoma[i]] for i, llave in enumerate(LLAVES)}

# ─── 2. INICIALIZACIÓN ────────────────────────────────────
def crear_cromosoma():
    return [random.randint(0, lon - 1) for lon in LONGITUDES]

def inicializar_poblacion(tam):
    return [crear_cromosoma() for _ in range(tam)]

# ─── 3. FUNCIÓN DE APTITUD ────────────────────────────────
def evaluar_aptitud(cromosoma):
    params = cromosoma_a_hiperparametros(cromosoma)
    modelo = RandomForestClassifier(**params, random_state=RANDOM_SEED, n_jobs=-1)
    scores = cross_val_score(modelo, X_scaled, y, cv=5, scoring='accuracy')
    return scores.mean()

# ─── 4. SELECCIÓN (Ruleta / Proporcional) ─────────────────
def seleccion_ruleta(poblacion, aptitudes):
    total = sum(aptitudes)
    if total == 0:
        return random.choice(poblacion)[:]
    pick = random.uniform(0, total)
    acumulado = 0
    for individuo, apt in zip(poblacion, aptitudes):
        acumulado += apt
        if acumulado >= pick:
            return individuo[:]
    return poblacion[-1][:]

# ─── 5. CRUZAMIENTO (Uniforme) ────────────────────────────
def cruzamiento_uniforme(padre1, padre2):
    if random.random() < TASA_CRUCE:
        mascara = [random.randint(0, 1) for _ in range(len(padre1))]
        hijo1 = [padre1[i] if mascara[i] else padre2[i] for i in range(len(padre1))]
        hijo2 = [padre2[i] if mascara[i] else padre1[i] for i in range(len(padre1))]
        return hijo1, hijo2
    return padre1[:], padre2[:]

# ─── 6. MUTACIÓN (Reemplazar gen por valor aleatorio) ─────
def mutar(cromosoma):
    for i in range(len(cromosoma)):
        if random.random() < TASA_MUTACION:
            cromosoma[i] = random.randint(0, LONGITUDES[i] - 1)
    return cromosoma

# ─── CICLO PRINCIPAL ──────────────────────────────────────
poblacion = inicializar_poblacion(TAM_POBLACION)
historial = []

print("\n  Evolucionando...")
mejor_global = None
mejor_aptitud_global = 0.0

for gen in range(NUM_GENERACIONES):
    aptitudes = [evaluar_aptitud(c) for c in poblacion]

    mejor_idx  = np.argmax(aptitudes)
    mejor_apt  = aptitudes[mejor_idx]
    historial.append(mejor_apt)

    if mejor_apt > mejor_aptitud_global:
        mejor_aptitud_global = mejor_apt
        mejor_global = poblacion[mejor_idx][:]

    if (gen + 1) % 5 == 0 or gen == 0:
        print(f"  Gen {gen+1:3d} | Mejor: {mejor_apt:.4f} | "
              f"Promedio: {np.mean(aptitudes):.4f}")

    # ─── 7. TERMINACIÓN al llegar a NUM_GENERACIONES ──────

    # Elitismo
    elite = poblacion[mejor_idx][:]
    nueva_pob = [elite]

    while len(nueva_pob) < TAM_POBLACION:
        p1 = seleccion_ruleta(poblacion, aptitudes)
        p2 = seleccion_ruleta(poblacion, aptitudes)
        h1, h2 = cruzamiento_uniforme(p1, p2)
        nueva_pob.append(mutar(h1))
        if len(nueva_pob) < TAM_POBLACION:
            nueva_pob.append(mutar(h2))

    poblacion = nueva_pob

# ─── RESULTADOS FINALES ───────────────────────────────────
mejores_params = cromosoma_a_hiperparametros(mejor_global)

# Comparar con hiperparámetros por defecto
default_modelo = RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1)
acc_default = cross_val_score(default_modelo, X_scaled, y, cv=5).mean()

print("\n" + "=" * 60)
print("  RESULTADO FINAL")
print("=" * 60)
print(f"  Accuracy AG (CV-5)      : {mejor_aptitud_global:.4f}")
print(f"  Accuracy default (CV-5) : {acc_default:.4f}")
print(f"  Mejora                  : +{(mejor_aptitud_global - acc_default)*100:.2f}%")
print("\n  Mejores hiperparámetros encontrados:")
for k, v in mejores_params.items():
    print(f"    {k:25s} = {v}")
print("=" * 60)

  AG PARA HYPERPARAMETER OPTIMIZATION — Wine Dataset
  Muestras    : 178
  Features    : 13
  Clases      : 3

  Evolucionando...
  Gen   1 | Mejor: 0.9722 | Promedio: 0.9607
  Gen   5 | Mejor: 0.9776 | Promedio: 0.9683
  Gen  10 | Mejor: 0.9778 | Promedio: 0.9690
  Gen  15 | Mejor: 0.9778 | Promedio: 0.9704
  Gen  20 | Mejor: 0.9778 | Promedio: 0.9686
  Gen  25 | Mejor: 0.9778 | Promedio: 0.9638

  RESULTADO FINAL
  Accuracy AG (CV-5)      : 0.9778
  Accuracy default (CV-5) : 0.9721
  Mejora                  : +0.57%

  Mejores hiperparámetros encontrados:
    n_estimators              = 50
    max_depth                 = None
    min_samples_split         = 2
    min_samples_leaf          = 2
    max_features              = sqrt


In [ ]:
"""
=============================================================
  ALGORITMO GENÉTICO PARA NEUROEVOLUTION
=============================================================
  Objetivo: Encontrar la mejor arquitectura de una red
  neuronal (MLP) usando un Algoritmo Genético.
  Dataset: Digits (sklearn) — reconocimiento de dígitos 0-9
  Se optimiza: número de capas, neuronas por capa,
               función de activación y tasa de aprendizaje.
=============================================================
"""

import numpy as np
import random
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

# ─── Configuración general ────────────────────────────────
RANDOM_SEED      = 42
TAM_POBLACION    = 14
NUM_GENERACIONES = 20
TASA_CRUCE       = 0.80
TASA_MUTACION    = 0.12

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ─── Carga del dataset ────────────────────────────────────
datos = load_digits()
X, y  = datos.data, datos.target
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("=" * 60)
print("  AG PARA NEUROEVOLUTION — Digits Dataset")
print("=" * 60)
print(f"  Muestras    : {X.shape[0]}")
print(f"  Features    : {X.shape[1]}  (imágenes 8x8)")
print(f"  Clases      : {len(np.unique(y))}  (dígitos 0–9)")
print("=" * 60)

# ─── 1. REPRESENTACIÓN (CROMOSOMA) ────────────────────────
# El cromosoma codifica la arquitectura de la red neuronal:
#
# Gen 0 : Número de capas ocultas  → índice en [1, 2, 3]
# Gen 1 : Neuronas capa 1          → índice en [32, 64, 128, 256]
# Gen 2 : Neuronas capa 2          → índice en [16, 32, 64, 128]
# Gen 3 : Neuronas capa 3          → índice en [8, 16, 32, 64]
# Gen 4 : Función de activación    → índice en ['relu','tanh','logistic']
# Gen 5 : Tasa de aprendizaje      → índice en [0.001, 0.005, 0.01, 0.05]
# Gen 6 : Algoritmo de optimización→ índice en ['adam','sgd','lbfgs']

ESPACIO_ARQ = {
    'num_capas'   : [1, 2, 3],
    'neuronas_c1' : [32, 64, 128, 256],
    'neuronas_c2' : [16, 32, 64, 128],
    'neuronas_c3' : [8, 16, 32, 64],
    'activacion'  : ['relu', 'tanh', 'logistic'],
    'lr'          : [0.001, 0.005, 0.01, 0.05],
    'solver'      : ['adam', 'sgd', 'lbfgs'],
}

LLAVES    = list(ESPACIO_ARQ.keys())
LONGITUDES = [len(ESPACIO_ARQ[k]) for k in LLAVES]

def cromosoma_a_red(cromosoma):
    """Traduce el cromosoma a parámetros de MLPClassifier."""
    params = {k: ESPACIO_ARQ[k][cromosoma[i]] for i, k in enumerate(LLAVES)}

    # Construir hidden_layer_sizes según num_capas activas
    capas = [params['neuronas_c1']]
    if params['num_capas'] >= 2:
        capas.append(params['neuronas_c2'])
    if params['num_capas'] == 3:
        capas.append(params['neuronas_c3'])

    return {
        'hidden_layer_sizes' : tuple(capas),
        'activation'         : params['activacion'],
        'learning_rate_init' : params['lr'],
        'solver'             : params['solver'],
        'max_iter'           : 300,
        'random_state'       : RANDOM_SEED,
    }

# ─── 2. INICIALIZACIÓN ────────────────────────────────────
def crear_cromosoma():
    return [random.randint(0, lon - 1) for lon in LONGITUDES]

def inicializar_poblacion(tam):
    return [crear_cromosoma() for _ in range(tam)]

# ─── 3. FUNCIÓN DE APTITUD ────────────────────────────────
def evaluar_aptitud(cromosoma):
    params = cromosoma_a_red(cromosoma)
    try:
        modelo = MLPClassifier(**params)
        scores = cross_val_score(modelo, X_scaled, y, cv=3, scoring='accuracy')
        return scores.mean()
    except Exception:
        return 0.0

# ─── 4. SELECCIÓN (Torneo doble) ──────────────────────────
def seleccion_torneo(poblacion, aptitudes, k=3):
    competidores = random.sample(range(len(poblacion)), k)
    ganador = max(competidores, key=lambda i: aptitudes[i])
    return poblacion[ganador][:]

# ─── 5. CRUZAMIENTO (Dos puntos) ──────────────────────────
def cruzamiento_dos_puntos(padre1, padre2):
    if random.random() < TASA_CRUCE and len(padre1) > 2:
        p1, p2 = sorted(random.sample(range(1, len(padre1)), 2))
        hijo1 = padre1[:p1] + padre2[p1:p2] + padre1[p2:]
        hijo2 = padre2[:p1] + padre1[p1:p2] + padre2[p2:]
        return hijo1, hijo2
    return padre1[:], padre2[:]

# ─── 6. MUTACIÓN (Reemplazo aleatorio de gen) ─────────────
def mutar(cromosoma):
    for i in range(len(cromosoma)):
        if random.random() < TASA_MUTACION:
            cromosoma[i] = random.randint(0, LONGITUDES[i] - 1)
    return cromosoma

# ─── CICLO PRINCIPAL ──────────────────────────────────────
poblacion  = inicializar_poblacion(TAM_POBLACION)
historial  = []
mejor_global = None
mejor_aptitud_global = 0.0

print("\n  Evolucionando arquitecturas neuronales...")
for gen in range(NUM_GENERACIONES):
    aptitudes = [evaluar_aptitud(c) for c in poblacion]

    mejor_idx = np.argmax(aptitudes)
    mejor_apt = aptitudes[mejor_idx]
    historial.append(mejor_apt)

    if mejor_apt > mejor_aptitud_global:
        mejor_aptitud_global = mejor_apt
        mejor_global = poblacion[mejor_idx][:]

    if (gen + 1) % 4 == 0 or gen == 0:
        print(f"  Gen {gen+1:3d} | Mejor: {mejor_apt:.4f} | "
              f"Promedio: {np.mean(aptitudes):.4f}")

    # ─── 7. TERMINACIÓN al completar NUM_GENERACIONES ─────

    # Elitismo (top-2)
    top2_idx = np.argsort(aptitudes)[-2:]
    nueva_pob = [poblacion[i][:] for i in top2_idx]

    while len(nueva_pob) < TAM_POBLACION:
        p1 = seleccion_torneo(poblacion, aptitudes)
        p2 = seleccion_torneo(poblacion, aptitudes)
        h1, h2 = cruzamiento_dos_puntos(p1, p2)
        nueva_pob.append(mutar(h1))
        if len(nueva_pob) < TAM_POBLACION:
            nueva_pob.append(mutar(h2))

    poblacion = nueva_pob

# ─── RESULTADOS FINALES ───────────────────────────────────
mejor_red = cromosoma_a_red(mejor_global)

# Comparar con arquitectura por defecto (100,)
default_mlp = MLPClassifier(max_iter=300, random_state=RANDOM_SEED)
acc_default = cross_val_score(default_mlp, X_scaled, y, cv=3).mean()

print("\n" + "=" * 60)
print("  RESULTADO FINAL — MEJOR ARQUITECTURA ENCONTRADA")
print("=" * 60)
print(f"  Accuracy AG   (CV-3)     : {mejor_aptitud_global:.4f}")
print(f"  Accuracy default (CV-3)  : {acc_default:.4f}")
print(f"  Mejora                   : +{(mejor_aptitud_global - acc_default)*100:.2f}%")
print("\n  Arquitectura óptima:")
print(f"    Capas ocultas          : {mejor_red['hidden_layer_sizes']}")
print(f"    Activación             : {mejor_red['activation']}")
print(f"    Tasa de aprendizaje    : {mejor_red['learning_rate_init']}")
print(f"    Optimizador            : {mejor_red['solver']}")
print("=" * 60)

  AG PARA NEUROEVOLUTION — Digits Dataset
  Muestras    : 1797
  Features    : 64  (imágenes 8x8)
  Clases      : 10  (dígitos 0–9)

  Evolucionando arquitecturas neuronales...


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptro

  Gen   1 | Mejor: 0.9432 | Promedio: 0.9274


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptro

  Gen   4 | Mejor: 0.9494 | Promedio: 0.9465


...

  Gen  12 | Mejor: 0.9505 | Promedio: 0.9443


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptro

  Gen  16 | Mejor: 0.9505 | Promedio: 0.9479


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


  Gen  20 | Mejor: 0.9505 | Promedio: 0.9495

  RESULTADO FINAL — MEJOR ARQUITECTURA ENCONTRADA
  Accuracy AG   (CV-3)     : 0.9505
  Accuracy default (CV-3)  : 0.9438
  Mejora                   : +0.67%

  Arquitectura óptima:
    Capas ocultas          : (256,)
    Activación             : relu
    Tasa de aprendizaje    : 0.05
    Optimizador            : sgd
